In [1]:
# NB23 CELL 1
# Purpose: Load primary model, scaler, and all TCGA data
# All four evaluations run on PRIMARY model only
# xgboost_final.pkl, 478 patients, C-index 0.702

import os
import pickle
import json
import numpy as np
import pandas as pd

os.chdir("/Users/parthshringarpure/Desktop/AI/Projects/luad_survival")

# --- Load primary model ---
with open("models/final_leakagefree/xgboost_final.pkl", "rb") as f:
    model_xgb = pickle.load(f)
with open("models/final_leakagefree/scaler_xgb.pkl", "rb") as f:
    scaler_xgb = pickle.load(f)

print(f"Primary model loaded: {type(model_xgb).__name__}")
print(f"Scaler loaded:        {type(scaler_xgb).__name__}")

# --- Load TCGA clinical ---
clin = pd.read_csv("data/processed/clinical_survival.csv", index_col=0)
print(f"\nTCGA clinical: {clin.shape}")
print(f"Columns: {list(clin.columns)}")

# --- Load TCGA expression (1000 genes) ---
expr = pd.read_csv("data/processed/expression_matrix.csv", index_col=0)
print(f"TCGA expression: {expr.shape}")

# --- Load dysregulation scores ---
dysreg = pd.read_csv("data/processed/dysregulation_scores.csv", index_col=0)
print(f"Dysregulation scores: {dysreg.shape}")

# --- Load immune features ---
immune = pd.read_csv(
    "data/processed/immune_features_cibersort.csv", index_col=0
)
print(f"Immune features: {immune.shape}")

# --- Load Lasso gene list ---
lasso_coef = pd.read_csv(
    "outputs/results/cox_lasso_coefficients.csv", index_col=0
)
lasso_genes = lasso_coef.index.tolist()
print(f"\nLasso genes: {len(lasso_genes)}")

# --- Align all datasets to 478 patients ---
patients = expr.index.tolist()
clin_478 = clin.loc[clin.index.intersection(patients)].copy()
clin_478 = clin_478.reindex(patients)

print(f"\nAligned patients: {len(patients)}")
print(f"Events: {clin_478['event'].sum()} "
      f"({100*clin_478['event'].mean():.1f}%)")
print(f"Stage distribution:\n"
      f"{clin_478['stage'].value_counts(dropna=False)}")

# --- Build structured survival array ---
y_time  = clin_478['survival_time'].values.astype(float)
y_event = clin_478['event'].values.astype(bool)

y_structured = np.array(
    [(bool(e), float(t)) for e, t in zip(y_event, y_time)],
    dtype=[('event', bool), ('time', float)]
)
print(f"\nStructured array: {y_structured.shape}")
print(f"Time range: {y_time.min():.0f} to {y_time.max():.0f} days")

# --- Check model feature expectations ---
try:
    n_features = model_xgb.n_features_in_
    print(f"\nModel expects {n_features} features")
except AttributeError:
    print("\nCannot determine n_features from model directly")

print("\n=== CELL 1 COMPLETE ===")
print("Next: Cell 2 — Rebuild feature matrix for primary model")

Primary model loaded: GradientBoostingSurvivalAnalysis
Scaler loaded:        StandardScaler

TCGA clinical: (484, 10)
Columns: ['vital_status', 'days_to_death', 'days_to_last_followup', 'age', 'gender', 'stage', 'survival_time', 'event', 'stage_group', 'age_group']
TCGA expression: (478, 1000)
Dysregulation scores: (478, 819)
Immune features: (478, 22)

Lasso genes: 72

Aligned patients: 478
Events: 121 (25.3%)
Stage distribution:
stage
stage ia      127
stage ib      124
stage iiia     67
stage iib      65
stage iia      46
stage iv       25
stage iiib     10
NaN             8
stage i         5
stage ii        1
Name: count, dtype: int64

Structured array: (478,)
Time range: 1 to 6812 days

Model expects 124 features

=== CELL 1 COMPLETE ===
Next: Cell 2 — Rebuild feature matrix for primary model


In [2]:
# NB23 CELL 2
# Purpose: Reconstruct the exact 124-feature matrix the primary model was trained on
# Strategy: build features incrementally, check shape matches 124, then verify
# with a quick C-index check against the known 0.702 result

from sksurv.metrics import concordance_index_censored
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# --- Normalise stage labels ---
def normalise_stage(s):
    if not isinstance(s, str):
        return None
    s = s.strip().lower().replace(' ', '')
    if s in ['stagei', 'stageia', 'stageib']:        return 'Stage I'
    elif s in ['stageii', 'stageiia', 'stageiib']:   return 'Stage II'
    elif s in ['stageiii', 'stageiiia', 'stageiiib']:return 'Stage III'
    elif s in ['stageiv']:                            return 'Stage IV'
    else:                                             return None

clin_478['stage_clean'] = clin_478['stage'].apply(normalise_stage)
print(f"Stage distribution after normalisation:")
print(clin_478['stage_clean'].value_counts(dropna=False))

# Stage dummies (Stage I as reference)
clin_478['stage_II']  = (clin_478['stage_clean'] == 'Stage II').astype(float)
clin_478['stage_III'] = (clin_478['stage_clean'] == 'Stage III').astype(float)
clin_478['stage_IV']  = (clin_478['stage_clean'] == 'Stage IV').astype(float)
age = clin_478['age'].fillna(clin_478['age'].median()).values

# --- Feature block 1: 72 Lasso expression genes ---
lasso_expr = expr[lasso_genes].values  # 478 x 72
print(f"\nExpression (72 Lasso genes): {lasso_expr.shape}")

# --- Feature block 2: top-20 dysreg by Cox p-value ---
# In NB11c this was selected inside fold — for full-data evaluation
# we select on all 478 patients (same as inference time)
from sksurv.linear_model import CoxPHSurvivalAnalysis

dysreg_scores = []
for j in range(dysreg.shape[1]):
    try:
        cox = CoxPHSurvivalAnalysis(alpha=0.1)
        cox.fit(dysreg.values[:, j:j+1], y_structured)
        dysreg_scores.append(abs(cox.coef_[0]))
    except Exception:
        dysreg_scores.append(0.0)

top20_idx   = np.argsort(dysreg_scores)[::-1][:20]
top20_genes = [dysreg.columns[i] for i in top20_idx]
dysreg_top20 = dysreg.values[:, top20_idx]  # 478 x 20
print(f"Dysreg top-20 genes: {top20_genes[:5]}...")

# --- Feature block 3: 5 interaction terms ---
m2   = immune['Macrophages M2'].values
cd8  = immune['T cells CD8'].values
treg = immune['T cells regulatory (Tregs)'].values
sIII = clin_478['stage_III'].values
sIV  = clin_478['stage_IV'].values

interactions = np.column_stack([
    sIII * m2,    # stageIII x M2
    sIV  * cd8,   # stageIV  x CD8
    age  * sIII,  # age      x stageIII
    sIII * treg,  # stageIII x Treg
    m2   * cd8,   # M2       x CD8
])
print(f"Interaction terms: {interactions.shape}")

# --- Feature block 4: clinical ---
clinical_feats = np.column_stack([
    age,
    clin_478['stage_II'].values,
    clin_478['stage_III'].values,
    clin_478['stage_IV'].values,
])
print(f"Clinical features: {clinical_feats.shape}")

# --- Feature block 5: immune (22 cell types) ---
immune_vals = immune.values  # 478 x 22
print(f"Immune features: {immune_vals.shape}")

# --- Try combinations to hit 124 features ---
# 72 expr + 20 dysreg + 5 interact + 4 clinical + 22 immune = 123
# Try adding gender
clin_478['gender_male'] = (
    clin_478['gender'].str.lower() == 'male'
).astype(float)
gender = clin_478['gender_male'].values.reshape(-1, 1)

X_123 = np.hstack([lasso_expr, dysreg_top20, interactions,
                   clinical_feats, immune_vals])
X_124 = np.hstack([lasso_expr, dysreg_top20, interactions,
                   clinical_feats, immune_vals, gender])

print(f"\nX_123 shape: {X_123.shape}")
print(f"X_124 shape: {X_124.shape}")
print(f"Model expects: 124 features")

# --- Test both against model ---
for label, X in [("X_123", X_123), ("X_124", X_124)]:
    try:
        X_s    = scaler_xgb.transform(X)
        risk   = model_xgb.predict(X_s)
        ci     = concordance_index_censored(y_event, y_time, risk)[0]
        print(f"{label}: C-index = {ci:.4f}  "
              f"{'<-- MATCHES 0.702' if abs(ci-0.702) < 0.01 else ''}")
    except ValueError as e:
        print(f"{label}: FAILED — {e}")

print("\n=== CELL 2 COMPLETE ===")

Stage distribution after normalisation:
stage_clean
Stage I      256
Stage II     112
Stage III     77
Stage IV      25
None           8
Name: count, dtype: int64

Expression (72 Lasso genes): (478, 72)
Dysreg top-20 genes: ['DKK1', 'NTSR1', 'PPP1R3G', 'IDO2', 'RHCG']...
Interaction terms: (478, 5)
Clinical features: (478, 4)
Immune features: (478, 22)

X_123 shape: (478, 123)
X_124 shape: (478, 124)
Model expects: 124 features
X_123: FAILED — X has 123 features, but StandardScaler is expecting 124 features as input.
X_124: C-index = 0.5776  

=== CELL 2 COMPLETE ===


/opt/anaconda3/envs/luad_survival/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/luad_survival/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/luad_survival/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingSurvivalAnalysis was fitted with feature names
  warnings.warn(


In [3]:
# NB23 CELL 3
# Purpose: Extract exact feature names and order from fitted scaler
# The scaler was fitted on a named DataFrame — feature_names_in_ has the answer

import numpy as np

# --- Get exact feature names from scaler ---
if hasattr(scaler_xgb, 'feature_names_in_'):
    feature_names = list(scaler_xgb.feature_names_in_)
    print(f"Scaler feature names: {len(feature_names)}")
    print(f"\nFirst 10: {feature_names[:10]}")
    print(f"\nLast 10:  {feature_names[-10:]}")
    print(f"\nAll 124 features:")
    for i, name in enumerate(feature_names):
        print(f"  {i+1:>3}. {name}")
else:
    print("No feature_names_in_ — checking model instead")
    if hasattr(model_xgb, 'feature_names_in_'):
        feature_names = list(model_xgb.feature_names_in_)
        print(f"Model feature names: {len(feature_names)}")
        for i, name in enumerate(feature_names):
            print(f"  {i+1:>3}. {name}")

Scaler feature names: 124

First 10: ['EPGN_expr', 'SLC47A1_expr', 'SIX1_expr', 'RHCG_expr', 'GPC6_expr', 'CCDC40_expr', 'CRHR2_expr', 'SH3TC2_expr', 'HS3ST2_expr', 'ACSM5_expr']

Last 10:  ['age', 'gender', 'stage_Stage II', 'stage_Stage III', 'stage_Stage IV', 'stageIII_x_M2', 'stageIV_x_CD8', 'age_x_stageIII', 'stageIII_x_Treg', 'M2_x_CD8']

All 124 features:
    1. EPGN_expr
    2. SLC47A1_expr
    3. SIX1_expr
    4. RHCG_expr
    5. GPC6_expr
    6. CCDC40_expr
    7. CRHR2_expr
    8. SH3TC2_expr
    9. HS3ST2_expr
   10. ACSM5_expr
   11. CASP14_expr
   12. TSPAN11_expr
   13. CNTN3_expr
   14. ALX1_expr
   15. TRPC2_expr
   16. BEND5_expr
   17. STK33_expr
   18. ERO1LB_expr
   19. IRF4_expr
   20. NAV3_expr
   21. TG_expr
   22. SERPINA6_expr
   23. CD70_expr
   24. LST-3TM12_expr
   25. MPV17L_expr
   26. LTK_expr
   27. NOS3_expr
   28. PDE10A_expr
   29. TMEM213_expr
   30. NLRP2_expr
   31. KCNA6_expr
   32. PCSK9_expr
   33. C8orf47_expr
   34. SLCO1B1_expr
   35. TMEM21

In [4]:
# NB23 CELL 4
# Purpose: Reconstruct exact 124-feature DataFrame matching scaler column order
# Then verify C-index matches 0.702 before running evaluations

import pandas as pd
import numpy as np
from sksurv.metrics import concordance_index_censored

feature_names = list(scaler_xgb.feature_names_in_)

# --- Parse feature groups from names ---
expr_feat_genes   = [f.replace('_expr',   '') for f in feature_names
                     if f.endswith('_expr')]
dysreg_feat_genes = [f.replace('_dysreg', '') for f in feature_names
                     if f.endswith('_dysreg')]
immune_feat_names = [f for f in feature_names
                     if f in immune.columns]

print(f"Expression genes:    {len(expr_feat_genes)}")
print(f"Dysreg genes:        {len(dysreg_feat_genes)}")
print(f"Immune cell types:   {len(immune_feat_names)}")

# --- Build each block as named Series/DataFrame ---

# Expression block (72 genes x 478 patients)
expr_block = expr[expr_feat_genes].copy()
expr_block.columns = [f"{g}_expr" for g in expr_feat_genes]

# Dysreg block (20 genes)
dysreg_block = dysreg[dysreg_feat_genes].copy()
dysreg_block.columns = [f"{g}_dysreg" for g in dysreg_feat_genes]

# Immune block (22 cell types, raw names)
immune_block = immune[immune_feat_names].copy()

# Clinical block
age_vals    = clin_478['age'].fillna(clin_478['age'].median())
gender_vals = (clin_478['gender'].str.lower() == 'male').astype(float)
stage_II    = (clin_478['stage_clean'] == 'Stage II').astype(float)
stage_III   = (clin_478['stage_clean'] == 'Stage III').astype(float)
stage_IV    = (clin_478['stage_clean'] == 'Stage IV').astype(float)

# Interaction terms
m2_vals   = immune['Macrophages M2'].values
cd8_vals  = immune['T cells CD8'].values
treg_vals = immune['T cells regulatory (Tregs)'].values
sIII_vals = stage_III.values
sIV_vals  = stage_IV.values
age_arr   = age_vals.values

interact_block = pd.DataFrame({
    'stageIII_x_M2'  : sIII_vals * m2_vals,
    'stageIV_x_CD8'  : sIV_vals  * cd8_vals,
    'age_x_stageIII' : age_arr   * sIII_vals,
    'stageIII_x_Treg': sIII_vals * treg_vals,
    'M2_x_CD8'       : m2_vals   * cd8_vals,
}, index=expr.index)

clin_block = pd.DataFrame({
    'age'           : age_arr,
    'gender'        : gender_vals.values,
    'stage_Stage II' : stage_II.values,
    'stage_Stage III': stage_III.values,
    'stage_Stage IV' : stage_IV.values,
}, index=expr.index)

# --- Assemble in exact order ---
X_full = pd.concat([
    expr_block,
    dysreg_block,
    immune_block,
    clin_block,
    interact_block
], axis=1)

# Reindex to exact scaler column order
X_full = X_full[feature_names]
print(f"\nAssembled feature matrix: {X_full.shape}")
print(f"Column order matches scaler: "
      f"{list(X_full.columns) == feature_names}")
print(f"NaNs: {X_full.isna().sum().sum()}")

# --- Scale and predict ---
X_scaled = scaler_xgb.transform(X_full)
risk_scores = model_xgb.predict(X_scaled)

ci = concordance_index_censored(y_event, y_time, risk_scores)[0]
print(f"\nReconstructed C-index: {ci:.4f}")
print(f"Expected:              0.702x")
print(f"Match: {'YES ✓' if abs(ci - 0.702) < 0.01 else 'NO — investigate'}")

# Save risk scores for evaluation cells
print(f"\nRisk score range: {risk_scores.min():.4f} to {risk_scores.max():.4f}")
print(f"Risk score mean:  {risk_scores.mean():.4f}")

print("\n=== CELL 4 COMPLETE ===")
print("Next: Cell 5 — Time-dependent AUC at 1, 3, 5 years")

Expression genes:    72
Dysreg genes:        20
Immune cell types:   22

Assembled feature matrix: (478, 124)
Column order matches scaler: True
NaNs: 0

Reconstructed C-index: 0.8890
Expected:              0.702x
Match: NO — investigate

Risk score range: -2.0732 to 4.2975
Risk score mean:  -0.0103

=== CELL 4 COMPLETE ===
Next: Cell 5 — Time-dependent AUC at 1, 3, 5 years


/opt/anaconda3/envs/luad_survival/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingSurvivalAnalysis was fitted with feature names
  warnings.warn(


In [6]:
# NB23 CELL 5
# Purpose: Fix DataFrame-based prediction, compute time-dependent AUC
# Pass DataFrame to scaler/model to suppress feature name warnings
# Then compute cumulative_dynamic_auc at 1, 3, 5 years

import numpy as np
import pandas as pd
from sksurv.metrics import cumulative_dynamic_auc
from sksurv.metrics import concordance_index_censored
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# --- Correct prediction: pass DataFrame not numpy array ---
X_scaled_df = pd.DataFrame(
    scaler_xgb.transform(X_full),
    columns=feature_names,
    index=X_full.index
)
risk_scores = model_xgb.predict(X_scaled_df)
print(f"Risk scores shape: {risk_scores.shape}")
print(f"Risk score range: {risk_scores.min():.4f} to {risk_scores.max():.4f}")

# In-sample C-index (expected ~0.88-0.92 — model on its own training data)
ci_insample = concordance_index_censored(y_event, y_time, risk_scores)[0]
print(f"In-sample C-index: {ci_insample:.4f}  "
      f"(training data — CV estimate is 0.702)")

# --- Time-dependent AUC ---
# cumulative_dynamic_auc requires:
#   survival_train: structured array (event, time) for training set
#   survival_test:  structured array for test set
#   estimate:       risk scores
#   times:          evaluation timepoints in same units as survival time
# Here train == test (full dataset evaluation)

# Timepoints: 1, 3, 5 years in days
times_years = np.array([1, 3, 5])
times_days  = times_years * 365.25

# Restrict to times within observed range
# cumulative_dynamic_auc requires times < max observed event time
max_event_time = y_time[y_event].max()
min_event_time = y_time[y_event].min()
print(f"\nObserved event time range: "
      f"{min_event_time:.0f} to {max_event_time:.0f} days")
print(f"Evaluation times (days): {times_days}")

# Check all times are within range
times_valid = times_days[
    (times_days > min_event_time) & (times_days < max_event_time)
]
print(f"Valid evaluation times: {times_valid} "
      f"({times_valid/365.25} years)")

# --- Compute time-dependent AUC ---
print(f"\nComputing time-dependent AUC...")
auc_values, mean_auc = cumulative_dynamic_auc(
    y_structured,   # training survival
    y_structured,   # test survival (same — full dataset)
    risk_scores,
    times_valid
)

print(f"\n=== TIME-DEPENDENT AUC RESULTS ===")
for t_days, auc in zip(times_valid, auc_values):
    t_years = t_days / 365.25
    print(f"  AUC at {t_years:.0f} year(s) "
          f"({t_days:.0f} days): {auc:.4f}")
print(f"  Mean AUC: {mean_auc:.4f}")

# --- AUC curve over time (dense timepoints) ---
# Evaluate at every 6 months from 6mo to max event time
times_curve = np.arange(182, max_event_time, 182)  # every ~6 months
times_curve = times_curve[times_curve < max_event_time]

auc_curve, _ = cumulative_dynamic_auc(
    y_structured,
    y_structured,
    risk_scores,
    times_curve
)

# --- Plot ---
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(times_curve / 365.25, auc_curve,
        color='steelblue', linewidth=2, label='Time-dependent AUC')
ax.axhline(0.5, color='grey', linestyle='--',
           linewidth=1, label='Random (0.5)')
ax.axhline(mean_auc, color='darkorange', linestyle='-.',
           linewidth=1.5, label=f'Mean AUC = {mean_auc:.3f}')

# Mark 1, 3, 5 year points
for t_days, auc in zip(times_valid, auc_values):
    t_years = t_days / 365.25
    ax.scatter(t_years, auc, color='red', zorder=5, s=80)
    ax.annotate(f'{auc:.3f}',
                xy=(t_years, auc),
                xytext=(t_years + 0.1, auc - 0.02),
                fontsize=10)

ax.set_xlabel('Time (years)', fontsize=12)
ax.set_ylabel('AUC', fontsize=12)
ax.set_title('Time-Dependent AUC — LUAD Survival Oracle\n'
             '(Primary Model, 478 TCGA patients)',
             fontsize=12)
ax.set_ylim(0.45, 1.0)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/figures/10_timedep_auc.png', dpi=150)
plt.close()
print(f"\nSaved: outputs/figures/10_timedep_auc.png")

print("\n=== CELL 5 COMPLETE ===")
print("Next: Cell 6 — Decision Curve Analysis")

Risk scores shape: (478,)
Risk score range: -2.0732 to 4.2975
In-sample C-index: 0.8890  (training data — CV estimate is 0.702)

Observed event time range: 4 to 4961 days
Evaluation times (days): [ 365.25 1095.75 1826.25]
Valid evaluation times: [ 365.25 1095.75 1826.25] ([1. 3. 5.] years)

Computing time-dependent AUC...

=== TIME-DEPENDENT AUC RESULTS ===
  AUC at 1 year(s) (365 days): 0.9103
  AUC at 3 year(s) (1096 days): 0.9506
  AUC at 5 year(s) (1826 days): 0.9188
  Mean AUC: 0.9303

Saved: outputs/figures/10_timedep_auc.png

=== CELL 5 COMPLETE ===
Next: Cell 6 — Decision Curve Analysis


In [7]:
# NB23 CELL 6
# Purpose: Decision Curve Analysis (DCA)
# Compare: our model vs treat-all vs treat-none vs clinical-only
# Net benefit at threshold probabilities 0.1 to 0.9
# Answers: is the model clinically useful?

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
import warnings
warnings.filterwarnings('ignore')

# --- Convert risk scores to 3-year mortality probabilities ---
# Use survival function from Cox model fitted on clinical only (baseline)
# and from our model's risk scores
# DCA needs predicted probability of event by time t

# Fit a simple Cox model to get baseline survival function
# Then use our risk scores to adjust per patient

# Step 1: Fit Cox on full data to get baseline hazard
from sksurv.linear_model import CoxPHSurvivalAnalysis

# Our model risk scores as single covariate
cox_calibrate = CoxPHSurvivalAnalysis(alpha=0.1)
risk_df = pd.DataFrame({'risk': risk_scores}, index=X_full.index)
cox_calibrate.fit(risk_df, y_structured)

# Predict survival functions for all patients
surv_funcs = cox_calibrate.predict_survival_function(risk_df)

# Extract 3-year (1096 day) survival probability per patient
t_eval = 1095.75  # 3 years in days
prob_death_3yr = np.array([
    1 - fn(t_eval) for fn in surv_funcs
])
print(f"3-year mortality probability range: "
      f"{prob_death_3yr.min():.4f} to {prob_death_3yr.max():.4f}")
print(f"Mean 3-year mortality: {prob_death_3yr.mean():.4f}")

# Observed 3-year mortality (for reference)
obs_3yr = y_event[y_time <= t_eval].mean() if (y_time <= t_eval).sum() > 0 else y_event.mean()
print(f"Observed event rate (all patients): {y_event.mean():.4f}")

# --- Clinical-only model probabilities ---
# Refit Cox on clinical features only
clin_feats = X_full[['age', 'gender',
                      'stage_Stage II',
                      'stage_Stage III',
                      'stage_Stage IV']].copy()
cox_clin = CoxPHSurvivalAnalysis(alpha=0.1)
cox_clin.fit(clin_feats, y_structured)
surv_clin = cox_clin.predict_survival_function(clin_feats)
prob_death_clin = np.array([1 - fn(t_eval) for fn in surv_clin])
print(f"\nClinical-only 3yr mortality range: "
      f"{prob_death_clin.min():.4f} to {prob_death_clin.max():.4f}")

# --- DCA function ---
def compute_net_benefit(prob_positive, y_true, thresholds):
    """
    prob_positive: predicted probability of event
    y_true:        observed event (bool array)
    thresholds:    array of decision thresholds
    Returns net_benefit array
    """
    n = len(y_true)
    net_benefits = []
    for thresh in thresholds:
        predicted_pos = prob_positive >= thresh
        tp = (predicted_pos & y_true).sum()
        fp = (predicted_pos & ~y_true).sum()
        nb = (tp / n) - (fp / n) * (thresh / (1 - thresh))
        net_benefits.append(nb)
    return np.array(net_benefits)

thresholds = np.arange(0.05, 0.95, 0.01)
n_patients = len(y_event)
event_rate = y_event.mean()

# Net benefit: our model
nb_model = compute_net_benefit(prob_death_3yr, y_event, thresholds)

# Net benefit: clinical only
nb_clin = compute_net_benefit(prob_death_clin, y_event, thresholds)

# Net benefit: treat all (predict everyone dies)
nb_treat_all = np.array([
    event_rate - (1 - event_rate) * (t / (1 - t))
    for t in thresholds
])

# Net benefit: treat none = 0
nb_treat_none = np.zeros(len(thresholds))

# --- Plot DCA ---
fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(thresholds, nb_model,
        color='steelblue', linewidth=2.5,
        label='LUAD Survival Oracle')
ax.plot(thresholds, nb_clin,
        color='darkorange', linewidth=2,
        linestyle='--', label='Clinical model (stage + age)')
ax.plot(thresholds, nb_treat_all,
        color='green', linewidth=1.5,
        linestyle=':', label='Treat all')
ax.plot(thresholds, nb_treat_none,
        color='grey', linewidth=1.5,
        linestyle='-.', label='Treat none')

ax.set_xlabel('Threshold Probability', fontsize=12)
ax.set_ylabel('Net Benefit', fontsize=12)
ax.set_title('Decision Curve Analysis — 3-Year Mortality\n'
             'LUAD Survival Oracle vs Clinical Baseline',
             fontsize=12)
ax.set_xlim(0.05, 0.90)
ax.set_ylim(-0.05, 0.35)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/figures/11_decision_curve.png', dpi=150)
plt.close()
print("Saved: outputs/figures/11_decision_curve.png")

# --- Report net benefit at clinically relevant thresholds ---
print(f"\n=== NET BENEFIT AT KEY THRESHOLDS ===")
print(f"{'Threshold':>10} {'Our Model':>12} {'Clinical':>12} "
      f"{'Treat All':>12} {'Difference':>12}")
print("-" * 62)
for thresh in [0.10, 0.20, 0.30, 0.40, 0.50]:
    idx = np.argmin(np.abs(thresholds - thresh))
    diff = nb_model[idx] - nb_clin[idx]
    print(f"{thresh:>10.2f} {nb_model[idx]:>12.4f} "
          f"{nb_clin[idx]:>12.4f} "
          f"{nb_treat_all[idx]:>12.4f} "
          f"{diff:>12.4f}")

# Clinical interpretation
print(f"\n=== CLINICAL INTERPRETATION ===")
# Find range where our model beats treat-all
beats_treat_all = thresholds[nb_model > nb_treat_all]
beats_clinical  = thresholds[nb_model > nb_clin]
if len(beats_treat_all) > 0:
    print(f"Model beats treat-all for thresholds: "
          f"{beats_treat_all.min():.2f} to {beats_treat_all.max():.2f}")
if len(beats_clinical) > 0:
    print(f"Model beats clinical-only for thresholds: "
          f"{beats_clinical.min():.2f} to {beats_clinical.max():.2f}")

print("\n=== CELL 6 COMPLETE ===")
print("Next: Cell 7 — Calibration curve")

3-year mortality probability range: 0.0124 to 1.0000
Mean 3-year mortality: 0.3642
Observed event rate (all patients): 0.2531

Clinical-only 3yr mortality range: 0.1330 to 0.8625
Saved: outputs/figures/11_decision_curve.png

=== NET BENEFIT AT KEY THRESHOLDS ===
 Threshold    Our Model     Clinical    Treat All   Difference
--------------------------------------------------------------
      0.10       0.1718       0.1702       0.1702       0.0016
      0.20       0.1067       0.0832       0.0664       0.0235
      0.30       0.0944       0.0430      -0.0669       0.0514
      0.40       0.0746      -0.0153      -0.2448       0.0900
      0.50       0.0690      -0.0690      -0.4937       0.1381

=== CLINICAL INTERPRETATION ===
Model beats treat-all for thresholds: 0.09 to 0.94
Model beats clinical-only for thresholds: 0.09 to 0.90

=== CELL 6 COMPLETE ===
Next: Cell 7 — Calibration curve


In [10]:
# NB23 CELL 7 (fix)
# Remove chi-square test — use MAE and visual calibration only
# Everything else from Cell 7 is correct

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# --- Calibration table already computed --- 
# pred_mean, obs_rate, bin_counts from Cell 7 are still in memory
# Just reprint table and save figure

print(f"Calibration bins: {len(pred_mean)}")
print(f"\n=== CALIBRATION TABLE (3-year mortality) ===")
print(f"{'Bin':>4} {'N':>6} {'Predicted':>10} {'Observed':>10} "
      f"{'Difference':>12}")
print("-" * 46)
for i, (pred, obs, n) in enumerate(
    zip(pred_mean, obs_rate, bin_counts)
):
    direction = 'over' if pred > obs else 'under'
    print(f"{i+1:>4} {n:>6} {pred:>10.4f} {obs:>10.4f} "
          f"{pred-obs:>+12.4f}  ({direction})")

cal_error = np.mean(np.abs(pred_mean - obs_rate))
cal_bias  = np.mean(pred_mean - obs_rate)  # positive = overestimates risk
print(f"\nMean absolute calibration error: {cal_error:.4f}")
print(f"Mean calibration bias:           {cal_bias:+.4f} "
      f"({'overestimates' if cal_bias > 0 else 'underestimates'} risk)")
print(f"Interpretation: "
      f"{'Good (<0.05)' if cal_error < 0.05 else 'Acceptable (0.05-0.10)' if cal_error < 0.10 else 'Poor (>0.10)'}")

# --- Calibration plot ---
fig, ax = plt.subplots(figsize=(7, 7))

ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5,
        label='Perfect calibration', alpha=0.7)

ax.scatter(pred_mean, obs_rate,
           s=[max(n/2, 20) for n in bin_counts],
           color='steelblue', alpha=0.8, zorder=5,
           label='LUAD Survival Oracle (deciles)')
ax.plot(pred_mean, obs_rate,
        color='steelblue', linewidth=1.5, alpha=0.6)

# Trend line
from scipy.interpolate import interp1d
if len(pred_mean) >= 4:
    sort_idx  = np.argsort(pred_mean)
    f_smooth  = interp1d(
        pred_mean[sort_idx], obs_rate[sort_idx],
        kind='linear', fill_value='extrapolate'
    )
    x_smooth = np.linspace(pred_mean.min(), pred_mean.max(), 100)
    ax.plot(x_smooth, f_smooth(x_smooth),
            color='darkorange', linewidth=2,
            linestyle='-', label='Trend', alpha=0.8)

ax.set_xlabel('Predicted 3-Year Mortality Probability', fontsize=12)
ax.set_ylabel('Observed 3-Year Mortality Rate', fontsize=12)
ax.set_title(
    'Calibration Curve — 3-Year Mortality Prediction\n'
    f'LUAD Survival Oracle (MAE = {cal_error:.3f}, '
    f'Bias = {cal_bias:+.3f})',
    fontsize=12
)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

ax.text(0.05, 0.92,
        f'MAE = {cal_error:.3f}\nBias = {cal_bias:+.3f}',
        transform=ax.transAxes, fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('outputs/figures/12_calibration_curve.png', dpi=150)
plt.close()
print("\nSaved: outputs/figures/12_calibration_curve.png")

print("\n=== CELL 7 (fix) COMPLETE ===")
print("Next: Cell 8 — Within-stage analysis")

Calibration bins: 10

=== CALIBRATION TABLE (3-year mortality) ===
 Bin      N  Predicted   Observed   Difference
----------------------------------------------
   1     48     0.0711     0.0000      +0.0711  (over)
   2     48     0.1240     0.0000      +0.1240  (over)
   3     48     0.1646     0.1429      +0.0218  (over)
   4     47     0.2112     0.5000      -0.2888  (under)
   5     48     0.2541     0.2000      +0.0541  (over)
   6     48     0.3042     0.2500      +0.0542  (over)
   7     47     0.3806     0.8333      -0.4528  (under)
   8     48     0.4835     0.8571      -0.3736  (under)
   9     48     0.6751     0.9583      -0.2832  (under)
  10     48     0.9707     1.0000      -0.0293  (under)

Mean absolute calibration error: 0.1753
Mean calibration bias:           -0.1102 (underestimates risk)
Interpretation: Poor (>0.10)

Saved: outputs/figures/12_calibration_curve.png

=== CELL 7 (fix) COMPLETE ===
Next: Cell 8 — Within-stage analysis


In [11]:
# NB23 CELL 8
# Purpose: Within-stage analysis
# Key question: does our molecular model add value WITHIN each stage
# that clinical staging alone cannot provide?
# Compare: our model C-index vs clinical-only C-index
# within Stage I patients only and Stage III patients only

import numpy as np
import pandas as pd
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# --- Stage group masks ---
stage_clean = clin_478['stage_clean'].values

mask_I   = stage_clean == 'Stage I'
mask_II  = stage_clean == 'Stage II'
mask_III = stage_clean == 'Stage III'
mask_IV  = stage_clean == 'Stage IV'

print(f"=== STAGE GROUP SIZES ===")
print(f"Stage I:   {mask_I.sum()} patients, "
      f"{y_event[mask_I].sum()} events")
print(f"Stage II:  {mask_II.sum()} patients, "
      f"{y_event[mask_II].sum()} events")
print(f"Stage III: {mask_III.sum()} patients, "
      f"{y_event[mask_III].sum()} events")
print(f"Stage IV:  {mask_IV.sum()} patients, "
      f"{y_event[mask_IV].sum()} events")

# --- Clinical-only Cox model ---
clin_feats = X_full[['age', 'gender',
                      'stage_Stage II',
                      'stage_Stage III',
                      'stage_Stage IV']].copy()
cox_clin = CoxPHSurvivalAnalysis(alpha=0.1)
cox_clin.fit(clin_feats, y_structured)
risk_clin = cox_clin.predict(clin_feats)

# --- Within-stage C-index function ---
def within_stage_cindex(risk, mask, y_ev, y_tm, label):
    """C-index within a stage group"""
    if mask.sum() < 10:
        return None, None
    r_sub  = risk[mask]
    ev_sub = y_ev[mask]
    tm_sub = y_tm[mask]
    if ev_sub.sum() < 3:
        print(f"  {label}: too few events ({ev_sub.sum()}) — skipped")
        return None, None
    ci = concordance_index_censored(ev_sub, tm_sub, r_sub)[0]
    return ci, ev_sub.sum()

# --- Compute C-indices for each stage group ---
print(f"\n=== WITHIN-STAGE C-INDEX ANALYSIS ===")
print(f"{'Stage':<12} {'N':>5} {'Events':>8} "
      f"{'Our Model':>12} {'Clinical':>12} {'Gain':>8}")
print("-" * 60)

results = []
for label, mask in [
    ('Stage I',   mask_I),
    ('Stage II',  mask_II),
    ('Stage III', mask_III),
    ('Stage IV',  mask_IV),
    ('All',       np.ones(478, dtype=bool)),
]:
    ci_model, n_ev = within_stage_cindex(
        risk_scores, mask, y_event, y_time, label
    )
    ci_clin, _     = within_stage_cindex(
        risk_clin, mask, y_event, y_time, label
    )
    if ci_model is None:
        continue

    gain = ci_model - ci_clin
    n    = mask.sum()
    print(f"{label:<12} {n:>5} {n_ev:>8} "
          f"{ci_model:>12.4f} {ci_clin:>12.4f} {gain:>+8.4f}")
    results.append({
        'stage'   : label,
        'n'       : n,
        'events'  : n_ev,
        'ci_model': ci_model,
        'ci_clin' : ci_clin,
        'gain'    : gain
    })

# --- Key biological interpretation ---
print(f"\n=== BIOLOGICAL INTERPRETATION ===")
for r in results:
    if r['stage'] == 'All':
        continue
    if r['gain'] > 0.02:
        print(f"Stage {r['stage'].split()[-1]}: "
              f"molecular model adds {r['gain']:+.4f} C-index — "
              f"molecular features refine prognosis beyond staging")
    elif r['gain'] < -0.02:
        print(f"Stage {r['stage'].split()[-1]}: "
              f"clinical model better by {abs(r['gain']):.4f} — "
              f"staging dominates in this group")
    else:
        print(f"Stage {r['stage'].split()[-1]}: "
              f"models equivalent (gain={r['gain']:+.4f})")

# --- Within-stage Kaplan-Meier by risk group ---
# Split each stage into high/low risk by median score
# This is the visual proof that molecular features work within stages
from sksurv.nonparametric import kaplan_meier_estimator
from scipy.stats import chi2_contingency
from lifelines.statistics import logrank_test

print(f"\n=== LOG-RANK TEST WITHIN EACH STAGE ===")
print(f"(High vs low molecular risk, split at median per stage)")
print(f"{'Stage':<12} {'N high':>8} {'N low':>8} {'p-value':>10}")
print("-" * 42)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
stage_labels = ['Stage I', 'Stage II', 'Stage III']
stage_masks  = [mask_I, mask_II, mask_III]

for ax, label, mask in zip(axes, stage_labels, stage_masks):
    if mask.sum() < 10:
        continue

    risk_sub  = risk_scores[mask]
    ev_sub    = y_event[mask]
    tm_sub    = y_time[mask]

    median_risk = np.median(risk_sub)
    high_mask   = risk_sub >= median_risk
    low_mask    = ~high_mask

    # Log-rank test
    lr = logrank_test(
        tm_sub[high_mask], tm_sub[low_mask],
        ev_sub[high_mask], ev_sub[low_mask]
    )
    pval = lr.p_value

    print(f"{label:<12} {high_mask.sum():>8} {low_mask.sum():>8} "
          f"{pval:>10.4f}")

    # KM curves
    t_high, s_high = kaplan_meier_estimator(
        ev_sub[high_mask], tm_sub[high_mask]
    )
    t_low, s_low = kaplan_meier_estimator(
        ev_sub[low_mask], tm_sub[low_mask]
    )

    ax.step(t_high / 365.25, s_high,
            color='red', linewidth=2, label='High risk')
    ax.step(t_low  / 365.25, s_low,
            color='blue', linewidth=2, label='Low risk')
    ax.set_title(f'{label}\n(p={pval:.4f})', fontsize=11)
    ax.set_xlabel('Time (years)', fontsize=10)
    ax.set_ylabel('Survival probability', fontsize=10)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    pval_str = f'p={pval:.4f}' if pval >= 0.0001 else 'p<0.0001'
    ax.text(0.05, 0.08, pval_str,
            transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle(
    'Within-Stage Survival Curves — Molecular Risk Stratification\n'
    'LUAD Survival Oracle (High vs Low Risk by Median Score)',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.savefig('outputs/figures/13_within_stage_km.png',
            dpi=150, bbox_inches='tight')
plt.close()
print(f"\nSaved: outputs/figures/13_within_stage_km.png")

print("\n=== CELL 8 COMPLETE ===")
print("NB23 COMPLETE — all four evaluations done")
print("Next: NB24 — Download and validate GSE31210 + TCGA-LUSC")

=== STAGE GROUP SIZES ===
Stage I:   256 patients, 39 events
Stage II:  112 patients, 34 events
Stage III: 77 patients, 35 events
Stage IV:  25 patients, 11 events

=== WITHIN-STAGE C-INDEX ANALYSIS ===
Stage            N   Events    Our Model     Clinical     Gain
------------------------------------------------------------
Stage I        256       39       0.8881       0.6189  +0.2693
Stage II       112       34       0.8402       0.4584  +0.3817
Stage III       77       35       0.8896       0.5667  +0.3229
Stage IV        25       11       0.7473       0.6099  +0.1374
All            478      121       0.8890       0.7012  +0.1878

=== BIOLOGICAL INTERPRETATION ===
Stage I: molecular model adds +0.2693 C-index — molecular features refine prognosis beyond staging
Stage II: molecular model adds +0.3817 C-index — molecular features refine prognosis beyond staging
Stage III: molecular model adds +0.3229 C-index — molecular features refine prognosis beyond staging
Stage IV: molecular mod